# Method 3 — GPU pairwise GBDT for business entity resolution

Supervised pairwise matching: **multi-core normalisation → GPU nearest-neighbour retrieval → pair features → XGBoost on GPU (hard negatives by construction) → F0.5-tuned decision policy**.

| Step | Runs on |
|---|---|
| 1. Load TSVs | CPU |
| 2. Normalise names / addresses (regex) | **all CPU cores** (multiprocessing) |
| 3. Character n-gram TF-IDF → dense embeddings | **GPU** (sparse × random projection) |
| 4. Candidate retrieval (name + address channels, per source) | **GPU** (matmul + top-k) |
| 5. Pair features | rapidfuzz on all cores + GPU cosines + vectorised code comparisons |
| 6–7. Labels, folds, XGBoost | **GPU** (`device="cuda"`) |
| 8. Decision policy (calibration, exclusivity, threshold / margin) | CPU, vectorised |
| 9–10. Test set + submission files | same pipeline |

**How to run:** Runtime → Change runtime type → **GPU** + **High-RAM** (the full dataset needs roughly 30 GB of system RAM) → set `DATA_DIR` below → Runtime → Run all.

`DATA_DIR` must contain `train/` (`train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`, `train_ground_truth.tsv`) and `test/` (`test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`).

In [ ]:
# ================================ settings ================================
DATA_DIR = "/content/dataset"               # folder that contains train/ and test/
OUT_DIR = "/content/outputs/method3_gbdt"   # submission files + report are written here
MOUNT_GOOGLE_DRIVE = False                  # True if DATA_DIR / OUT_DIR are on Drive (/content/drive/MyDrive/...)

N_FOLDS = 3          # out-of-fold XGBoost models; their average also scores the test set
K_PER_SOURCE = 10    # candidates kept per S1 record from S2 and from S3 (15 = a bit more recall, +50% time / memory)
RETRIEVE_K = 20      # nearest neighbours per retrieval channel before fusion (>= K_PER_SOURCE)
EMB_DIM = 256        # retrieval embedding size
SEED = 42

In [ ]:
import os

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

for split in ("train", "test"):
    path = os.path.join(DATA_DIR, split)
    assert os.path.isdir(path), f"missing folder: {path}"
    print(path, sorted(os.listdir(path)))

In [ ]:
%pip install -q "rapidfuzz==3.13.0" "xgboost>=2.0"

In [ ]:
import csv
import gc
import json
import re
import time
import unicodedata
import warnings
from dataclasses import dataclass
from functools import lru_cache
from multiprocessing import get_context

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler, Levenshtein
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_JOBS = os.cpu_count() or 2
os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - T0:7.0f}s] {msg}", flush=True)


if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    log(f"GPU: {torch.cuda.get_device_name(0)}  ({free / 2**30:.1f} of {total / 2**30:.1f} GB free) | CPU cores: {N_JOBS}")
else:
    log(f"WARNING: no GPU visible (Runtime > Change runtime type > GPU). Running on {N_JOBS} CPU cores - much slower.")
log(f"xgboost {xgb.__version__}, torch {torch.__version__}")

## Step 1 — Load data

All columns are read as strings with NA-parsing off, so a business literally named "NA" stays "NA". If a stray double quote swallows lines, the file is re-read with quoting disabled.

In [ ]:
RECORD_COLUMNS = ["entity_id", "business_name", "business_address", "country"]


def read_tsv(path: str) -> pd.DataFrame:
    with open(path, "rb") as fh:
        expected = sum(1 for line in fh if line.strip()) - 1
    for quoting in (csv.QUOTE_MINIMAL, csv.QUOTE_NONE):
        df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False, quoting=quoting)
        if len(df) == expected:
            df.columns = [c.strip() for c in df.columns]
            return df
    raise ValueError(f"{path}: parsed {len(df)} rows, expected {expected}")


def load_records(path: str, prefix: str) -> pd.DataFrame:
    df = read_tsv(path)
    missing = [c for c in RECORD_COLUMNS if c not in df.columns]
    assert not missing, f"{path}: missing columns {missing}"
    df = df[RECORD_COLUMNS].apply(lambda s: s.str.strip())
    assert df.entity_id.is_unique, f"{path}: duplicate entity_id"
    assert df.entity_id.str.startswith(prefix).all(), f"{path}: ids without prefix {prefix}"
    return df.reset_index(drop=True)


def load_split(split: str) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    base = os.path.join(DATA_DIR, split)
    s1 = load_records(f"{base}/{split}_source1.tsv", "S1-")
    s2 = load_records(f"{base}/{split}_source2.tsv", "S2-")
    s3 = load_records(f"{base}/{split}_source3.tsv", "S3-")
    is_s3 = np.r_[np.zeros(len(s2), np.int8), np.ones(len(s3), np.int8)]
    return s1, pd.concat([s2, s3], ignore_index=True), is_s3


def load_ground_truth(path: str, s1_ids: pd.Series, s23_ids: pd.Series) -> np.ndarray:
    """Sorted int64 codes  s1_row * N23 + s23_row  of all true pairs."""
    gt = read_tsv(path)
    ex = pd.DataFrame({"s1": gt.source1_entity_id.str.strip(), "c": gt.matched_entity_ids.str.split(",")}).explode("c")
    ex["c"] = ex.c.fillna("").str.strip()
    ex = ex[ex.c != ""]
    gi = pd.Index(s1_ids).get_indexer(ex.s1)
    gj = pd.Index(s23_ids).get_indexer(ex.c)
    assert (gi >= 0).all() and (gj >= 0).all(), "ground truth references unknown ids"
    return np.unique(gi.astype(np.int64) * len(s23_ids) + gj)

In [ ]:
tr_s1, tr_s23, tr_is_s3 = load_split("train")
tr_gt = load_ground_truth(os.path.join(DATA_DIR, "train", "train_ground_truth.tsv"), tr_s1.entity_id, tr_s23.entity_id)
N1, N23 = len(tr_s1), len(tr_s23)
owners = np.bincount((tr_gt % N23).astype(np.int64)).max() if len(tr_gt) else 0
log(f"train: {N1:,} S1 | {N23:,} S2+S3 | {len(tr_gt):,} true pairs | max S1 owners per S2/S3 record = {owners}")

## Step 2 — Normalise names and addresses (all CPU cores)

Canonical tokens on both sides (`Pvt`/`Private` → `pvt`, `Rd`/`Road` → `rd`, `Shree`/`Sri` → `sri`, state names → codes, renamed cities), legal suffixes stripped from the core name, trade names after `dba` split off, landmark phrases ("Near SBI ATM") separated, postal code / house number / numbers extracted, country as an open set. This is regex work, so it runs on every CPU core in parallel (the GPU cannot help here).

**Lexicons:**

In [ ]:
_SHARED_MAP: dict[str, str] = {
    'saint': 'st', 'sainte': 'ste', 'mount': 'mt', 'fort': 'ft', 'sri': 'sri', 'shri': 'sri',
    'shree': 'sri', 'sree': 'sri', 'shrii': 'sri', 'shreee': 'sri', 'smt': 'smt', 'shrimati': 'smt',
    'srimati': 'smt', 'mohd': 'mohd', 'mohammed': 'mohd', 'mohammad': 'mohd', 'muhammad': 'mohd',
    'mohamed': 'mohd', 'muhammed': 'mohd', 'mohamad': 'mohd', 'mahmood': 'mahmud', 'mahmud': 'mahmud',
    'and': 'and',
}

_NAME_MAP: dict[str, str] = {
    **_SHARED_MAP, 'private': 'pvt', 'pvt': 'pvt', 'pte': 'pvt', 'limited': 'ltd', 'ltd': 'ltd',
    'ltda': 'ltd', 'corporation': 'corp', 'corpn': 'corp', 'corp': 'corp', 'incorporated': 'inc',
    'inc': 'inc', 'company': 'co', 'companies': 'co', 'cos': 'co', 'compagnie': 'co', 'cie': 'co',
    'international': 'intl', 'intl': 'intl', 'internatl': 'intl', 'national': 'natl', 'natl': 'natl',
    'manufacturing': 'mfg', 'manufacturers': 'mfg', 'manufacturer': 'mfg', 'mfrs': 'mfg', 'mfg': 'mfg',
    'services': 'svc', 'service': 'svc', 'svcs': 'svc', 'svc': 'svc', 'serv': 'svc', 'associates': 'assoc',
    'association': 'assoc', 'assn': 'assoc', 'assoc': 'assoc', 'brothers': 'bros', 'bros': 'bros',
    'freres': 'bros', 'enterprises': 'ent', 'enterprise': 'ent', 'entp': 'ent', 'ent': 'ent',
    'hospital': 'hosp', 'hospitals': 'hosp', 'hosp': 'hosp', 'center': 'ctr', 'centre': 'ctr',
    'cntr': 'ctr', 'ctr': 'ctr', 'department': 'dept', 'dept': 'dept', 'university': 'univ', 'univ': 'univ',
    'institute': 'inst', 'inst': 'inst', 'technologies': 'tech', 'technology': 'tech', 'techs': 'tech',
    'tech': 'tech', 'systems': 'sys', 'system': 'sys', 'sys': 'sys', 'management': 'mgmt', 'mgmt': 'mgmt',
    'development': 'dev', 'dev': 'dev', 'industries': 'ind', 'industry': 'ind', 'inds': 'ind', 'ind': 'ind',
    'market': 'mkt', 'markets': 'mkt', 'mkt': 'mkt', 'doctor': 'dr', 'dr': 'dr',
    'pharmaceuticals': 'pharma', 'pharmaceutical': 'pharma', 'pharma': 'pharma', 'medical': 'med',
    'med': 'med', 'group': 'grp', 'grp': 'grp', 'holdings': 'hldg', 'hldgs': 'hldg', 'hldg': 'hldg',
    'financial': 'fin', 'finance': 'fin', 'fin': 'fin', 'solutions': 'soln', 'solns': 'soln',
    'soln': 'soln', 'consultants': 'consult', 'consultancy': 'consult', 'consulting': 'consult',
    'consult': 'consult', 'engineering': 'eng', 'engg': 'eng', 'eng': 'eng', 'electricals': 'elec',
    'electrical': 'elec', 'electric': 'elec', 'elec': 'elec', 'travels': 'travel', 'travel': 'travel',
    'traders': 'trade', 'trader': 'trade', 'trading': 'trade', 'automobiles': 'auto', 'automobile': 'auto',
    'automotive': 'auto', 'auto': 'auto', 'motors': 'motor', 'motor': 'motor', 'laboratories': 'lab',
    'laboratory': 'lab', 'labs': 'lab', 'lab': 'lab', 'clinics': 'clinic', 'clinic': 'clinic',
    'agencies': 'agency', 'agency': 'agency', 'insurance': 'ins', 'ins': 'ins', 'communications': 'comm',
    'communication': 'comm', 'comm': 'comm', 'products': 'prod', 'product': 'prod', 'prod': 'prod',
    'restaurants': 'restaurant', 'hotels': 'hotel', 'stores': 'store', 'shoppe': 'shop',
    'etablissements': 'ets', 'etablissement': 'ets', 'ets': 'ets', 'societe': 'ste', 'ste': 'ste',
}

_ADDR_MAP: dict[str, str] = {
    **_SHARED_MAP, 'road': 'rd', 'rd': 'rd', 'street': 'st', 'st': 'st', 'str': 'st', 'stt': 'st',
    'avenue': 'ave', 'ave': 'ave', 'av': 'ave', 'avn': 'ave', 'boulevard': 'blvd', 'blvd': 'blvd',
    'bd': 'blvd', 'boul': 'blvd', 'bld': 'blvd', 'lane': 'ln', 'ln': 'ln', 'drive': 'dr', 'dr': 'dr',
    'drv': 'dr', 'court': 'ct', 'ct': 'ct', 'place': 'pl', 'pl': 'pl', 'square': 'sq', 'sq': 'sq',
    'highway': 'hwy', 'hwy': 'hwy', 'hiway': 'hwy', 'parkway': 'pkwy', 'pkwy': 'pkwy', 'expressway': 'expy',
    'expy': 'expy', 'terrace': 'ter', 'terr': 'ter', 'circle': 'cir', 'cir': 'cir', 'crescent': 'cres',
    'cres': 'cres', 'plaza': 'plz', 'plz': 'plz', 'suite': 'ste', 'suit': 'ste', 'apartment': 'apt',
    'apt': 'apt', 'floor': 'fl', 'flr': 'fl', 'fl': 'fl', 'building': 'bldg', 'bldg': 'bldg',
    'bldng': 'bldg', 'room': 'rm', 'rm': 'rm', 'north': 'n', 'south': 's', 'east': 'e', 'west': 'w',
    'northeast': 'ne', 'northwest': 'nw', 'southeast': 'se', 'southwest': 'sw', 'nagar': 'nagar',
    'ngr': 'nagar', 'nagr': 'nagar', 'marg': 'marg', 'mrg': 'marg', 'colony': 'colony', 'clny': 'colony',
    'col': 'colony', 'sector': 'sec', 'sect': 'sec', 'sec': 'sec', 'phase': 'phase', 'ph': 'phase',
    'block': 'blk', 'blk': 'blk', 'cross': 'cross', 'crs': 'cross', 'layout': 'layout', 'lyt': 'layout',
    'extension': 'extn', 'extn': 'extn', 'ext': 'extn', 'bazaar': 'bazar', 'bazar': 'bazar', 'bzr': 'bazar',
    'chowk': 'chowk', 'chk': 'chowk', 'mohalla': 'mohalla', 'mohala': 'mohalla', 'enclave': 'enclave',
    'encl': 'enclave', 'estate': 'estate', 'est': 'estate', 'industrial': 'indl', 'indl': 'indl',
    'plot': 'plot', 'plt': 'plot', 'opposite': 'opp', 'opp': 'opp', 'near': 'near', 'nr': 'near',
    'behind': 'behind', 'bhd': 'behind', 'junction': 'jn', 'jn': 'jn', 'jct': 'jn', 'junc': 'jn',
    'station': 'stn', 'stn': 'stn', 'railway': 'rly', 'rly': 'rly', 'district': 'dist', 'distt': 'dist',
    'dist': 'dist', 'village': 'vill', 'vill': 'vill', 'vil': 'vill', 'taluka': 'taluk', 'taluk': 'taluk',
    'chemin': 'chemin', 'ch': 'chemin', 'impasse': 'imp', 'imp': 'imp', 'route': 'rte', 'rte': 'rte',
    'faubourg': 'fbg', 'fbg': 'fbg', 'first': '1', 'second': '2', 'third': '3', 'fourth': '4', 'fifth': '5',
    'sixth': '6', 'seventh': '7', 'eighth': '8', 'ninth': '9', 'tenth': '10', 'bombay': 'mumbai',
    'madras': 'chennai', 'calcutta': 'kolkata', 'bangalore': 'bengaluru', 'gurgaon': 'gurugram',
    'poona': 'pune', 'trivandrum': 'thiruvananthapuram', 'baroda': 'vadodara', 'benares': 'varanasi',
    'banaras': 'varanasi', 'benaras': 'varanasi', 'cochin': 'kochi', 'mysore': 'mysuru',
    'mangalore': 'mangaluru', 'allahabad': 'prayagraj', 'pondicherry': 'puducherry', 'simla': 'shimla',
    'calicut': 'kozhikode', 'belgaum': 'belagavi', 'vizag': 'visakhapatnam', 'nyc': 'ny', 'alabama': 'al',
    'alaska': 'ak', 'arizona': 'az', 'arkansas': 'ar', 'california': 'ca', 'colorado': 'co',
    'connecticut': 'ct', 'delaware': 'de', 'florida': 'fl', 'georgia': 'ga', 'hawaii': 'hi', 'idaho': 'id',
    'illinois': 'il', 'indiana': 'in', 'iowa': 'ia', 'kansas': 'ks', 'kentucky': 'ky', 'louisiana': 'la',
    'maine': 'me', 'maryland': 'md', 'massachusetts': 'ma', 'michigan': 'mi', 'minnesota': 'mn',
    'mississippi': 'ms', 'missouri': 'mo', 'montana': 'mt', 'nebraska': 'ne', 'nevada': 'nv', 'ohio': 'oh',
    'oklahoma': 'ok', 'oregon': 'or', 'pennsylvania': 'pa', 'tennessee': 'tn', 'texas': 'tx', 'utah': 'ut',
    'vermont': 'vt', 'virginia': 'va', 'washington': 'wa', 'wisconsin': 'wi', 'wyoming': 'wy',
    'maharashtra': 'mh', 'karnataka': 'ka', 'kerala': 'kl', 'gujarat': 'gj', 'rajasthan': 'rj',
    'bihar': 'br', 'odisha': 'od', 'orissa': 'od', 'telangana': 'ts', 'punjab': 'pb', 'haryana': 'hr',
    'jharkhand': 'jh', 'chhattisgarh': 'cg', 'uttarakhand': 'uk', 'assam': 'as', 'goa': 'ga',
}

_ADDR_PHRASES: tuple[tuple[str, str], ...] = (
    ("mahatma gandhi", "mg"),
    ("new york", "ny"),
    ("new jersey", "nj"),
    ("new mexico", "nm"),
    ("new hampshire", "nh"),
    ("north carolina", "nc"),
    ("south carolina", "sc"),
    ("north dakota", "nd"),
    ("south dakota", "sd"),
    ("west virginia", "wv"),
    ("rhode island", "ri"),
    ("district of columbia", "dc"),
    ("uttar pradesh", "up"),
    ("madhya pradesh", "mp"),
    ("andhra pradesh", "ap"),
    ("himachal pradesh", "hp"),
    ("arunachal pradesh", "ar"),
    ("tamil nadu", "tn"),
    ("west bengal", "wb"),
    ("jammu and kashmir", "jk"),
    ("post office", "po"),
    ("p o", "po"),
    ("united states of america", ""),
    ("united states", ""),
)

_ADDR_PHRASE_RES = tuple((re.compile(rf"\b{p}\b"), r) for p, r in _ADDR_PHRASES)

LEGAL_TOKENS: frozenset[str] = frozenset(
    {
        "pvt",
        "ltd",
        "llc",
        "llp",
        "inc",
        "corp",
        "co",
        "plc",
        "lp",
        "pllc",
        "pc",
        "gmbh",
        "sarl",
        "sas",
        "sasu",
        "sa",
        "eurl",
        "snc",
        "sci",
        "ag",
        "bv",
        "nv",
        "opc",
        "lllp",
    }
)

_NAME_TAIL_EXTRA: frozenset[str] = frozenset({"india", "usa", "us", "america", "france", "and"})

_NAME_STOP: frozenset[str] = frozenset({"the", "and", "of", "le", "la", "les", "de", "du", "des", "et"})

_ADDR_DROP: frozenset[str] = frozenset({"india", "usa", "france", "cedex"})

_NUMBER_WORDS: frozenset[str] = frozenset({"no", "number", "num", "nos"})

_COUNTRY_ALIASES: dict[str, str] = {
    'us': 'us', 'usa': 'us', 'u s': 'us', 'u s a': 'us', 'united states': 'us',
    'united states of america': 'us', 'america': 'us', 'in': 'in', 'ind': 'in', 'india': 'in',
    'bharat': 'in', 'republic of india': 'in', 'fr': 'fr', 'fra': 'fr', 'france': 'fr',
    'republique francaise': 'fr', 'french republic': 'fr',
}

_LANDMARK_RE = re.compile(
    r"\b(?:near|nr|opp|opposite|behind|beside|besides|next to|adjacent to|adj to|adj|in front of|"
    r"infront of|close to|facing|across from|across)\b\.?(?:[ \t]+[^\s,;]+){0,2}"
)

_DBA_RE = re.compile(r"\b(?:d b a|dba|doing business as|t a|trading as|aka|a k a)\b")

_APOS_RE = re.compile(r"[’‘'`´]")

_ZIP4_RE = re.compile(r"\b(\d{5})\s*-\s*\d{4}\b")

_PIN_SPLIT_RE = re.compile(r"\b(\d{3})\s(\d{3})\s*$")

_ORDINAL_RE = re.compile(r"\b(\d+)(?:st|nd|rd|th)\b")

_PUNCT_RE = re.compile(r"[^\w\s]|_")

_ALNUM_SPLIT_RE = re.compile(r"(?<=[^\W\d_])(?=\d)|(?<=\d)(?=[^\W\d_])")

_WS_RE = re.compile(r"\s+")

_POSTAL_RE = re.compile(r"^\d{5,6}$")

_DIGITS_RE = re.compile(r"\d+")

_VOWELS = frozenset("aeiou")

**Normalisation functions:**

In [ ]:
def fold_unicode(text: str) -> str:
    """NFKD + strip combining marks (é -> e, ç -> c) + lower-case."""
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch)).lower()


def _clean(text: str) -> str:
    text = fold_unicode(text).replace("&", " and ").replace("@", " at ")
    text = _APOS_RE.sub("", text)
    text = _ZIP4_RE.sub(r"\1", text)
    text = _ORDINAL_RE.sub(r"\1", text)
    text = _PUNCT_RE.sub(" ", text)
    text = _ALNUM_SPLIT_RE.sub(" ", text)
    return _WS_RE.sub(" ", text).strip()


def _join_initials(tokens: list[str]) -> list[str]:
    """``s b i`` -> ``sbi`` (initialisms written with dots/spaces)."""
    out: list[str] = []
    run: list[str] = []
    for tok in tokens:
        if len(tok) == 1 and tok.isalpha():
            run.append(tok)
            continue
        if len(run) >= 2:
            out.append("".join(run))
        else:
            out.extend(run)
        run = []
        out.append(tok)
    if len(run) >= 2:
        out.append("".join(run))
    else:
        out.extend(run)
    return out


def _map_tokens(tokens: list[str], mapping: dict[str, str]) -> list[str]:
    return [m for m in (mapping.get(t, t) for t in tokens) if m]


def skeleton(token: str) -> str:
    """Transliteration-tolerant consonant skeleton: ``shree``/``sri`` -> ``sr``, ``baalaji`` -> ``blj``."""
    t = token
    for a, b in (
        ("ph", "f"),
        ("sh", "s"),
        ("kh", "k"),
        ("gh", "g"),
        ("th", "t"),
        ("dh", "d"),
        ("bh", "b"),
        ("jh", "j"),
        ("ch", "c"),
        ("ck", "k"),
        ("q", "k"),
        ("w", "v"),
        ("z", "j"),
        ("x", "ks"),
        ("y", "i"),
    ):
        t = t.replace(a, b)
    if not t:
        return t
    head, rest = t[0], "".join(ch for ch in t[1:] if ch not in _VOWELS)
    out = [head]
    for ch in rest:
        if ch != out[-1]:
            out.append(ch)
    return "".join(out)


def normalize_country(value: str) -> str:
    key = _WS_RE.sub(" ", _PUNCT_RE.sub(" ", fold_unicode(value))).strip()
    return _COUNTRY_ALIASES.get(key, key)


@dataclass(frozen=True)
class NormName:
    norm: str  # all tokens, canonicalised
    core: str  # legal suffixes / country tail / stop words removed
    alt: str  # trade name after "dba", "" if none
    legal: str  # sorted legal tokens found
    skel: str  # consonant skeleton of core tokens
    acronym: str  # initials of core tokens (>= 2 tokens), "" otherwise


def _strip_tail(tokens: list[str]) -> tuple[list[str], list[str]]:
    legal: list[str] = []
    end = len(tokens)
    while end > 1 and (tokens[end - 1] in LEGAL_TOKENS or tokens[end - 1] in _NAME_TAIL_EXTRA):
        if tokens[end - 1] in LEGAL_TOKENS:
            legal.append(tokens[end - 1])
        end -= 1
    return tokens[:end], legal


def _core_tokens(tokens: list[str]) -> tuple[list[str], list[str]]:
    body, legal = _strip_tail(tokens)
    legal += [t for t in body if t in LEGAL_TOKENS and t not in {"co", "sa", "ag", "pc"}]
    core = [t for t in body if t not in _NAME_STOP and not (t in LEGAL_TOKENS and t != body[0])]
    return (core or body), legal


@lru_cache(maxsize=1 << 16)  # bounded per worker process
def normalize_name(raw: str) -> NormName:
    cleaned = _clean(raw)
    parts = _DBA_RE.split(cleaned, maxsplit=1)
    main = _map_tokens(_join_initials(parts[0].split()), _NAME_MAP)
    alt_tokens = _map_tokens(_join_initials(parts[1].split()), _NAME_MAP) if len(parts) > 1 else []
    core, legal = _core_tokens(main) if main else ([], [])
    alt_core = _core_tokens(alt_tokens)[0] if alt_tokens else []
    acronym = "".join(t[0] for t in core if t) if len(core) >= 2 else ""
    return NormName(
        norm=" ".join(main + alt_tokens),
        core=" ".join(core),
        alt=" ".join(alt_core),
        legal=" ".join(sorted(set(legal))),
        skel=" ".join(skeleton(t) for t in core),
        acronym=acronym,
    )


@dataclass(frozen=True)
class NormAddress:
    norm: str  # full canonical address
    core: str  # landmark phrase + postal code removed
    landmark: str  # canonical landmark phrase ("near sbi atm"), "" if none
    postal: str  # 5-6 digit postal/ZIP/PIN code, "" if none
    nums: str  # space-joined sorted unique digit runs (postal excluded)
    house: str  # first number of the core address, "" if none


def _canon_address_tokens(text: str) -> list[str]:
    text = _clean(text)
    for rx, rep in _ADDR_PHRASE_RES:
        text = rx.sub(rep, text)
    toks = _join_initials(text.split())
    toks = [
        t for i, t in enumerate(toks) if not (t in _NUMBER_WORDS and i + 1 < len(toks) and toks[i + 1][:1].isdigit())
    ]
    return [t for t in _map_tokens(toks, _ADDR_MAP) if t not in _ADDR_DROP]


def _find_postal(tokens: list[str]) -> str:
    """Last 5-6 digit token. A leading one is usually a (US) house number, so it only counts as
    postal when it is 6 digits or directly followed by another number (reordered components,
    e.g. ``33005 50 impasse voltaire``)."""
    for i in range(len(tokens) - 1, 0, -1):
        if _POSTAL_RE.match(tokens[i]):
            return tokens[i]
    if tokens and _POSTAL_RE.match(tokens[0]):
        if len(tokens[0]) == 6 or (len(tokens) > 1 and tokens[1].isdigit()):
            return tokens[0]
    return ""


@lru_cache(maxsize=1 << 16)  # bounded per worker process
def normalize_address(raw: str) -> NormAddress:
    folded = fold_unicode(raw)
    folded = _PIN_SPLIT_RE.sub(r"\1\2", folded.strip())
    landmark_raw = " ".join(m.group(0) for m in _LANDMARK_RE.finditer(folded))
    without_landmark = _LANDMARK_RE.sub(" ", folded)

    full = _canon_address_tokens(folded)
    core = _canon_address_tokens(without_landmark)
    landmark = _canon_address_tokens(landmark_raw) if landmark_raw else []

    postal = _find_postal(core)
    if postal:
        core = [t for t in core if t != postal]
    digits = [t for t in core if t.isdigit()]
    nums = sorted(set(digits), key=lambda x: (len(x), x))
    return NormAddress(
        norm=" ".join(full),
        core=" ".join(core),
        landmark=" ".join(landmark),
        postal=postal,
        nums=" ".join(nums),
        house=digits[0] if digits else "",
    )

In [ ]:
NORM_FIELDS = ("name_core", "name_alt", "name_skel", "name_acronym", "name_legal",
               "addr_core", "addr_norm", "postal", "nums", "house", "country")
_SRC: tuple[list[str], list[str], list[str]] | None = None  # inherited by forked workers (no pickling of inputs)


def _normalize_range(bounds: tuple[int, int]) -> dict[str, list[str]]:
    lo, hi = bounds
    names, addrs, countries = _SRC
    out: dict[str, list[str]] = {f: [] for f in NORM_FIELDS}
    for i in range(lo, hi):
        n, a = normalize_name(names[i]), normalize_address(addrs[i])
        out["name_core"].append(n.core)
        out["name_alt"].append(n.alt)
        out["name_skel"].append(n.skel)
        out["name_acronym"].append(n.acronym)
        out["name_legal"].append(n.legal)
        out["addr_core"].append(a.core)
        out["addr_norm"].append(a.norm)
        out["postal"].append(a.postal)
        out["nums"].append(a.nums)
        out["house"].append(a.house)
        out["country"].append(normalize_country(countries[i]))
    return out


def normalize_frame(df: pd.DataFrame, desc: str) -> pd.DataFrame:
    global _SRC
    _SRC = (df.business_name.tolist(), df.business_address.tolist(), df.country.tolist())
    n, step = len(df), 20_000
    bounds = [(s, min(s + step, n)) for s in range(0, n, step)]
    cols: dict[str, list[str]] = {f: [] for f in NORM_FIELDS}
    with get_context("fork").Pool(N_JOBS) as pool, tqdm(total=n, desc=desc, unit="rec", unit_scale=True) as pb:
        for part in pool.imap(_normalize_range, bounds):
            for f in NORM_FIELDS:
                cols[f].extend(part[f])
            pb.update(len(part["name_core"]))
    _SRC = None
    return pd.DataFrame(cols)

In [ ]:
tr_n1 = normalize_frame(tr_s1, "normalise train S1")
tr_n23 = normalize_frame(tr_s23, "normalise train S2+S3")
tr_s1, tr_s23 = tr_s1[["entity_id"]], tr_s23[["entity_id"]]  # raw text no longer needed
gc.collect()
log("train normalised")
tr_n1.head()

## Step 3 — Dense retrieval embeddings (GPU)

Each record gets two embeddings: **name** (char 2–4-grams of core name + trade name) and **address** (char 3–4-grams of core address + postal code). Grams are hashed into 2¹⁸ TF-IDF dimensions on all CPU cores (IDF from a training sample, reused for test) and multiplied on the GPU by a fixed random Gaussian projection to `EMB_DIM` dimensions, then L2-normalised — a Johnson–Lindenstrauss sketch, so dot products approximate the TF-IDF cosine. Stored as float16.

In [ ]:
HASH_DIM = 2**18
VECTORIZERS = {
    "name": HashingVectorizer(analyzer="char_wb", ngram_range=(2, 4), n_features=HASH_DIM, alternate_sign=False,
                              norm=None, lowercase=False, dtype=np.float32),
    "addr": HashingVectorizer(analyzer="char_wb", ngram_range=(3, 4), n_features=HASH_DIM, alternate_sign=False,
                              norm=None, lowercase=False, dtype=np.float32),
}
_gen = torch.Generator().manual_seed(SEED)
PROJ = {ch: (torch.randn(HASH_DIM, EMB_DIM, generator=_gen) / EMB_DIM**0.5).to(DEVICE) for ch in VECTORIZERS}


def channel_texts(norm: pd.DataFrame, channel: str) -> list[str]:
    if channel == "name":
        return (norm.name_core + " " + norm.name_alt).str.strip().tolist()
    return (norm.addr_core + " " + norm.postal).str.strip().tolist()


def fit_idf(texts: list[str], channel: str) -> np.ndarray:
    x = VECTORIZERS[channel].transform(texts)
    df = np.bincount(x.indices, minlength=HASH_DIM).astype(np.float64)
    return (np.log((1 + x.shape[0]) / (1 + df)) + 1).astype(np.float32)


_TXT: tuple[list[str], str, np.ndarray] | None = None  # (texts, channel, idf) inherited by forked workers


def _hash_chunk(bounds: tuple[int, int]):
    """CPU worker: hashed sublinear TF-IDF rows for texts[lo:hi] (CSR parts)."""
    texts, channel, idf = _TXT
    x = VECTORIZERS[channel].transform(texts[bounds[0]:bounds[1]]).tocsr()
    x.data = (1.0 + np.log(x.data)) * idf[x.indices]
    return x.indptr.astype(np.int64), x.indices.astype(np.int64), x.data, x.shape


def embed(texts: list[str], channel: str, idf: np.ndarray, desc: str, chunk: int = 100_000) -> np.ndarray:
    """n-gram hashing on all CPU cores, projection + normalisation on the GPU."""
    global _TXT
    _TXT = (texts, channel, idf)
    out = np.empty((len(texts), EMB_DIM), dtype=np.float16)
    bounds = [(s, min(s + chunk, len(texts))) for s in range(0, len(texts), chunk)]
    with get_context("fork").Pool(N_JOBS) as pool:
        for (lo, hi), (indptr, indices, data, shape) in zip(
            bounds, tqdm(pool.imap(_hash_chunk, bounds), total=len(bounds), desc=desc, unit="chunk")
        ):
            xt = torch.sparse_csr_tensor(torch.from_numpy(indptr), torch.from_numpy(indices), torch.from_numpy(data),
                                         size=shape).to(DEVICE)
            with torch.no_grad():
                e = xt @ PROJ[channel]
                e = e / e.norm(dim=1, keepdim=True).clamp_min(1e-12)
            out[lo:hi] = e.half().cpu().numpy()
    _TXT = None
    return out


def embed_split(n1: pd.DataFrame, n23: pd.DataFrame, idf: dict[str, np.ndarray], split: str):
    e1 = {ch: embed(channel_texts(n1, ch), ch, idf[ch], f"{split} S1 {ch} embeddings") for ch in VECTORIZERS}
    e23 = {ch: embed(channel_texts(n23, ch), ch, idf[ch], f"{split} S2+S3 {ch} embeddings") for ch in VECTORIZERS}
    return e1, e23

In [ ]:
rng = np.random.default_rng(SEED)
IDF = {}
for ch in VECTORIZERS:
    t1, t23 = channel_texts(tr_n1, ch), channel_texts(tr_n23, ch)
    sample = [t1[i] for i in rng.choice(len(t1), min(300_000, len(t1)), replace=False)]
    sample += [t23[i] for i in rng.choice(len(t23), min(300_000, len(t23)), replace=False)]
    IDF[ch] = fit_idf(sample, ch)
tr_e1, tr_e23 = embed_split(tr_n1, tr_n23, IDF, "train")
log(f"train embeddings: S1 {tr_e1['name'].shape}, S2+S3 {tr_e23['name'].shape} (float16)")

## Step 4 — Candidate retrieval (GPU)

For every S1 record and each source separately (S2, then S3): top-`RETRIEVE_K` neighbours in three channels — **name**, **address**, and **name + address combined** (records similar in both fields, which is what separates branches of a chain) — each one GPU matmul + top-k per chunk of queries. The union is re-ranked by name cosine + address cosine + a reciprocal-rank bonus across the three channels, and the best `K_PER_SOURCE` per source are kept. Every S2/S3 record is searched — nothing is sampled. The per-channel ranks and cosines become features.

In [ ]:
def _query_chunk(n_docs: int) -> int:
    if DEVICE == "cuda":
        free, _ = torch.cuda.mem_get_info()
        return int(max(32, min(8192, free * 0.25 / (n_docs * 2 * 2))))
    return int(max(16, min(4096, 2e8 / max(n_docs, 1))))


def _rank_in(cands: torch.Tensor, ranked: torch.Tensor) -> torch.Tensor:
    """1-based position of each candidate in ``ranked`` (row-wise), len(ranked)+1 if absent."""
    hit = cands.unsqueeze(2) == ranked.unsqueeze(1)
    pos = hit.float().argmax(dim=2) + 1
    return torch.where(hit.any(dim=2), pos, torch.full_like(pos, ranked.shape[1] + 1))


@torch.no_grad()
def retrieve(e1: dict, e23: dict, is_s3: np.ndarray, split: str) -> dict[str, np.ndarray]:
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    n1 = len(e1["name"])
    parts = {k: [] for k in ("i", "j", "name_cos", "addr_cos", "rank_name", "rank_addr", "rank_full")}
    for src in (0, 1):
        rows = np.flatnonzero(is_s3 == src)
        if len(rows) == 0:
            continue
        kk = min(RETRIEVE_K, len(rows))
        kp = min(K_PER_SOURCE, kk)
        dn = torch.from_numpy(e23["name"][rows]).to(DEVICE, dtype)
        da = torch.from_numpy(e23["addr"][rows]).to(DEVICE, dtype)
        dfull = dn + da  # combined channel: finds records similar in name AND address across the whole corpus
        rows_t = torch.from_numpy(rows).to(DEVICE)
        chunk = _query_chunk(len(rows))
        for s in tqdm(range(0, n1, chunk), desc=f"{split}: retrieve vs {'S3' if src else 'S2'}", unit="chunk"):
            qn = torch.from_numpy(e1["name"][s:s + chunk]).to(DEVICE, dtype)
            qa = torch.from_numpy(e1["addr"][s:s + chunk]).to(DEVICE, dtype)
            top_n = torch.topk(qn @ dn.T, kk, dim=1).indices
            top_a = torch.topk(qa @ da.T, kk, dim=1).indices
            top_f = torch.topk((qn + qa) @ dfull.T, kk, dim=1).indices
            cand = torch.sort(torch.cat([top_n, top_a, top_f], dim=1), dim=1).values
            dup = torch.zeros_like(cand, dtype=torch.bool)
            dup[:, 1:] = cand[:, 1:] == cand[:, :-1]
            ncos = (qn.unsqueeze(1) * dn[cand]).sum(-1).float()
            acos = (qa.unsqueeze(1) * da[cand]).sum(-1).float()
            ranks = [_rank_in(cand, t) for t in (top_n, top_a, top_f)]
            rrf = sum(torch.where(r <= kk, 1.0 / (10.0 + r), torch.zeros_like(r, dtype=torch.float)) for r in ranks)
            fused = (ncos + acos + 5.0 * rrf).masked_fill(dup, float("-inf"))  # best cut rule on held-out recall
            best = torch.topk(fused, kp, dim=1)
            ok = torch.isfinite(best.values)
            sel = best.indices
            c = torch.gather(cand, 1, sel)
            qi = torch.arange(s, s + len(qn), device=DEVICE).unsqueeze(1).expand_as(c)
            for key, val in (("i", qi), ("j", rows_t[c]), ("name_cos", torch.gather(ncos, 1, sel)),
                             ("addr_cos", torch.gather(acos, 1, sel)), ("rank_name", torch.gather(ranks[0], 1, sel)),
                             ("rank_addr", torch.gather(ranks[1], 1, sel)), ("rank_full", torch.gather(ranks[2], 1, sel))):
                parts[key].append(val[ok].cpu().numpy())
        del dn, da, dfull, rows_t
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    out = {k: np.concatenate(v) for k, v in parts.items()}
    order = np.lexsort((out["j"], out["i"]))
    return {k: v[order] for k, v in out.items()}


def candidate_recall(pairs: dict, gt_codes: np.ndarray, n1: int, n23: int) -> dict[str, float]:
    codes = pairs["i"] * n23 + pairs["j"]
    found = np.isin(gt_codes, codes)
    n_true = np.bincount(gt_codes // n23, minlength=n1)
    n_found = np.bincount((gt_codes // n23)[found], minlength=n1)
    return {"pair_recall": float(found.mean()) if len(found) else 1.0,
            "s1_with_all_matches_found": float((n_found == n_true).mean()),
            "candidates_per_s1": len(codes) / n1}

In [ ]:
tr_pairs = retrieve(tr_e1, tr_e23, tr_is_s3, "train")
del tr_e1, tr_e23
gc.collect()
log(f"train candidates: {len(tr_pairs['i']):,} pairs | {candidate_recall(tr_pairs, tr_gt, N1, N23)}")

## Step 5 — Pair features

Records are first **encoded**: strings kept only where fuzzy matching needs them, everything compared for equality (postal, house number, country, legal form, acronym, numbers) turned into integer codes so comparisons are vectorised. Then, per chunk of pairs: rapidfuzz scorers (C++, all cores), equality / conflict flags, numeric-token overlap, name-frequency (chain) features. Finally **context features** over the whole pair table: each candidate's rank and gap within its S1 record, and how many S1 records compete for the same S2/S3 record (reverse rank) — the strongest signal against chain-store false merges.

In [ ]:
def _codes(*arrays: pd.Series) -> list[np.ndarray]:
    """Joint integer codes for several string columns ("" -> -1)."""
    values = pd.concat([a.reset_index(drop=True) for a in arrays], ignore_index=True)
    codes, _ = pd.factorize(values)
    codes = codes.astype(np.int32)
    codes[(values == "").to_numpy()] = -1
    out, s = [], 0
    for a in arrays:
        out.append(codes[s:s + len(a)])
        s += len(a)
    return out


def _nums_codes(a: pd.Series, b: pd.Series, width: int = 4) -> tuple[np.ndarray, np.ndarray]:
    tokens = pd.concat([a, b], ignore_index=True).str.split(" ", n=width, expand=True).iloc[:, :width]
    tokens = tokens.fillna("").reindex(columns=range(width), fill_value="")
    flat = tokens.to_numpy().ravel()
    codes, _ = pd.factorize(flat)
    codes = codes.astype(np.int32)
    codes[flat == ""] = -1
    codes = codes.reshape(-1, width)
    return codes[: len(a)], codes[len(a):]


def encode(n1: pd.DataFrame, n23: pd.DataFrame) -> tuple[dict, dict]:
    e1, e23 = {}, {}
    for col in ("name_core", "name_alt", "name_skel", "addr_core", "addr_norm"):
        e1[col], e23[col] = n1[col].to_numpy(object), n23[col].to_numpy(object)
    for col in ("postal", "house", "country", "name_legal"):
        e1[col], e23[col] = _codes(n1[col], n23[col])
    e1["postal3"], e23["postal3"] = _codes(n1.postal.str[:3], n23.postal.str[:3])
    # names and acronyms share one vocabulary so "SBI" can match the initials of "State Bank of India"
    e1["name_code"], e23["name_code"], e1["acr_code"], e23["acr_code"] = _codes(n1.name_core, n23.name_core,
                                                                               n1.name_acronym, n23.name_acronym)
    e1["nums"], e23["nums"] = _nums_codes(n1.nums, n23.nums)
    for e, n in ((e1, n1), (e23, n23)):
        e["name_len"] = n.name_core.str.len().to_numpy(np.float32)
        e["name_ntok"] = n.name_core.str.count(" ").to_numpy(np.float32) + (n.name_core != "").to_numpy(np.float32)
        e["addr_ntok"] = n.addr_core.str.count(" ").to_numpy(np.float32) + (n.addr_core != "").to_numpy(np.float32)
    n_codes = int(max(e1["name_code"].max(initial=-1), e23["name_code"].max(initial=-1))) + 1
    cnt1 = np.bincount(e1["name_code"][e1["name_code"] >= 0], minlength=n_codes)
    cnt23 = np.bincount(e23["name_code"][e23["name_code"] >= 0], minlength=n_codes)
    for e, cnt_self in ((e1, cnt1), (e23, cnt23)):
        code = e["name_code"]
        e["freq_self"] = np.where(code >= 0, np.log1p(cnt_self[np.maximum(code, 0)]), 0).astype(np.float32)
        e["freq_other"] = np.where(code >= 0, np.log1p((cnt23 if e is e1 else cnt1)[np.maximum(code, 0)]), 0).astype(np.float32)
    return e1, e23

In [ ]:
SCORERS = {"ratio": (fuzz.ratio, 100.0), "partial": (fuzz.partial_ratio, 100.0), "tsort": (fuzz.token_sort_ratio, 100.0),
           "tset": (fuzz.token_set_ratio, 100.0), "jw": (JaroWinkler.normalized_similarity, 1.0),
           "lev": (Levenshtein.normalized_similarity, 1.0)}

# features whose effect on P(match) must be non-decreasing (monotone constraints: robust on the unseen country)
MONOTONE = {"name_cos", "addr_cos", "name_ratio", "name_tsort", "name_tset", "name_jw", "addr_tset", "addr_tsort",
            "addr_ratio", "postal", "house", "country"}


def fz(a: np.ndarray, b: np.ndarray, scorer: str) -> np.ndarray:
    fn, scale = SCORERS[scorer]
    out = process.cpdist(a, b, scorer=fn, workers=-1, dtype=np.float32) / np.float32(scale)
    return np.asarray(out, dtype=np.float32)


def tri(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """1 equal, 0 different, NaN if missing on either side."""
    return np.where((a < 0) | (b < 0), np.nan, (a == b).astype(np.float32)).astype(np.float32)


def pair_features_chunk(e1: dict, e23: dict, i: np.ndarray, j: np.ndarray, is_s3: np.ndarray, p: dict) -> dict:
    f: dict[str, np.ndarray] = {}
    empty_n = (e1["name_len"][i] == 0) | (e23["name_len"][j] == 0)
    empty_a = (e1["addr_ntok"][i] == 0) | (e23["addr_ntok"][j] == 0)
    na, nb = e1["name_core"][i], e23["name_core"][j]
    for sc in ("ratio", "tsort", "tset", "partial", "jw", "lev"):
        f[f"name_{sc}"] = np.where(empty_n, np.nan, fz(na, nb, sc))
    f["skel_ratio"] = fz(e1["name_skel"][i], e23["name_skel"][j], "ratio")
    alt_a, alt_b = e1["name_alt"][i], e23["name_alt"][j]
    has_alt = np.flatnonzero((alt_a != "") | (alt_b != ""))
    alt = np.full(len(i), np.nan, np.float32)
    if len(has_alt):
        alt[has_alt] = np.fmax(fz(alt_a[has_alt], nb[has_alt], "tsort"), fz(na[has_alt], alt_b[has_alt], "tsort"))
    f["alt_best"] = alt
    aa, ab = e1["addr_core"][i], e23["addr_core"][j]
    for sc in ("tset", "tsort", "ratio", "partial"):
        f[f"addr_{sc}"] = np.where(empty_a, np.nan, fz(aa, ab, sc))
    f["addr_full_tset"] = fz(e1["addr_norm"][i], e23["addr_norm"][j], "tset")
    f["name_cos"], f["addr_cos"] = p["name_cos"], p["addr_cos"]
    for r in ("rank_name", "rank_addr", "rank_full"):
        f[r] = p[r].astype(np.float32)
    f["is_s3"] = is_s3[j].astype(np.float32)
    f["name_exact"] = tri(e1["name_code"][i], e23["name_code"][j])
    f["acronym"] = ((e1["acr_code"][i] >= 0) & (e1["acr_code"][i] == e23["name_code"][j])
                    | (e23["acr_code"][j] >= 0) & (e23["acr_code"][j] == e1["name_code"][i])).astype(np.float32)
    for col in ("postal", "postal3", "house", "country", "name_legal"):
        f[col] = tri(e1[col][i], e23[col][j])
    A, B = e1["nums"][i], e23["nums"][j]
    common = ((A[:, :, None] == B[:, None, :]) & (A[:, :, None] >= 0)).any(axis=2).sum(axis=1).astype(np.float32)
    la, lb = (A >= 0).sum(axis=1).astype(np.float32), (B >= 0).sum(axis=1).astype(np.float32)
    f["nums_common"], f["nums_only_s1"], f["nums_only_c"] = common, la - common, lb - common
    f["nums_jaccard"] = np.where((la == 0) | (lb == 0), np.nan, common / np.maximum(la + lb - common, 1))
    f["name_len_diff"] = np.abs(e1["name_len"][i] - e23["name_len"][j])
    f["name_ntok_diff"] = np.abs(e1["name_ntok"][i] - e23["name_ntok"][j])
    f["addr_ntok_s1"], f["addr_ntok_c"] = e1["addr_ntok"][i], e23["addr_ntok"][j]
    f["name_freq_s1"], f["name_freq_s1_in_c"] = e1["freq_self"][i], e1["freq_other"][i]
    f["name_freq_c"] = e23["freq_self"][j]
    with np.errstate(all="ignore"):
        f["name_score"] = np.nan_to_num(np.nanmean(np.vstack([f["name_cos"], f["name_tsort"], f["name_jw"]]), axis=0))
        f["addr_score"] = np.nanmean(np.vstack([f["addr_cos"], f["addr_tset"]]), axis=0)
    f["name_x_addr"] = f["name_score"] * f["addr_score"]
    f["addr_missing"] = empty_a.astype(np.float32)
    return f


def _group_top2(g: np.ndarray, v: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.lexsort((-v, g))
    gs, vs = g[order], v[order]
    first = np.r_[True, gs[1:] != gs[:-1]]
    starts = np.flatnonzero(first)
    sizes = np.diff(np.r_[starts, len(gs)])
    top = vs[starts]
    second = np.where(sizes > 1, vs[np.minimum(starts + 1, len(vs) - 1)], -np.inf)
    gid = np.cumsum(first) - 1
    t, s = np.empty(len(v)), np.empty(len(v))
    t[order], s[order] = top[gid], second[gid]
    return t, s


def context_features(i: np.ndarray, j: np.ndarray, name: np.ndarray, addr: np.ndarray, n1: int, n23: int) -> dict:
    combo = 0.5 * name + 0.5 * np.nan_to_num(addr, nan=0.5)
    s_max, s_2nd = _group_top2(i, combo)
    c_max, c_2nd = _group_top2(j, combo)
    other_cand = np.where(combo >= s_max, s_2nd, s_max)
    other_s1 = np.where(combo >= c_max, c_2nd, c_max)
    clip = lambda x: np.clip(x, -2, 2).astype(np.float32)  # noqa: E731  (-inf when there is no competitor)
    return {
        "combo": combo.astype(np.float32),
        "ctx_rank_in_s1": pd.Series(combo).groupby(i).rank(ascending=False, method="min").to_numpy(np.float32),
        "ctx_gap_to_s1_best": clip(combo - s_max),
        "ctx_margin_over_other_cand": clip(combo - other_cand),
        "ctx_n_strong_names": np.bincount(i, weights=name >= 0.9, minlength=n1)[i].astype(np.float32),
        "ctx_n_strong_addrs": np.bincount(i, weights=np.nan_to_num(addr) >= 0.9, minlength=n1)[i].astype(np.float32),
        "ctx_rank_in_cand": pd.Series(combo).groupby(j).rank(ascending=False, method="min").to_numpy(np.float32),
        "ctx_n_s1_competing": np.bincount(j, minlength=n23)[j].astype(np.float32),
        "ctx_margin_over_other_s1": clip(combo - other_s1),
    }


def build_features(e1: dict, e23: dict, pairs: dict, is_s3: np.ndarray, split: str,
                   chunk: int = 1_000_000) -> tuple[np.ndarray, list[str]]:
    n = len(pairs["i"])
    names, X = None, None
    for s in tqdm(range(0, n, chunk), desc=f"{split}: pair features", unit="chunk"):
        sl = slice(s, min(s + chunk, n))
        p = {k: v[sl] for k, v in pairs.items()}
        f = pair_features_chunk(e1, e23, p["i"], p["j"], is_s3, p)
        if X is None:
            names = list(f) + ["combo", "ctx_rank_in_s1", "ctx_gap_to_s1_best", "ctx_margin_over_other_cand",
                               "ctx_n_strong_names", "ctx_n_strong_addrs", "ctx_rank_in_cand", "ctx_n_s1_competing",
                               "ctx_margin_over_other_s1"]
            X = np.empty((n, len(names)), dtype=np.float32)
        for k, v in f.items():
            X[sl, names.index(k)] = v
    ctx = context_features(pairs["i"], pairs["j"], X[:, names.index("name_score")], X[:, names.index("addr_score")],
                           len(e1["name_core"]), len(e23["name_core"]))
    for k, v in ctx.items():
        X[:, names.index(k)] = v
    return X, names

In [ ]:
TR_COUNTRY = tr_n1.country.to_numpy()
tr_enc1, tr_enc23 = encode(tr_n1, tr_n23)
del tr_n1, tr_n23
gc.collect()
X, FEATURES = build_features(tr_enc1, tr_enc23, tr_pairs, tr_is_s3, "train")
log(f"train feature matrix: {X.shape[0]:,} pairs x {X.shape[1]} features ({X.nbytes / 2**30:.1f} GB)")

## Step 6 — Labels and folds

Training pairs are exactly the retrieved candidates, labelled with the ground truth: every negative already looks like a match in at least one channel (same-name branch, same-building neighbour, near-identical name) — the hard negatives the model has to learn. Folds are over S1 records, stratified by country × {0, 1, 2+} true matches.

In [ ]:
PI, PJ = tr_pairs["i"], tr_pairs["j"]
y = np.isin(PI * N23 + PJ, tr_gt)
N_TRUE = np.bincount(tr_gt // N23, minlength=N1)
COUNTRY1, COUNTRY_NAMES = pd.factorize(TR_COUNTRY, sort=True)


def make_folds(n1: int, strata: np.ndarray, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    folds = np.empty(n1, dtype=np.int64)
    offset = 0
    for code in np.unique(strata):
        members = np.flatnonzero(strata == code)
        folds[members[rng.permutation(len(members))]] = (np.arange(len(members)) + offset) % k
        offset += len(members)
    return folds


FOLD = make_folds(N1, COUNTRY1.astype(np.int64) * 3 + np.minimum(N_TRUE, 2), N_FOLDS, SEED)
PAIR_FOLD = FOLD[PI]
log(f"positives: {int(y.sum()):,} of {len(y):,} pairs ({y.mean():.4%}); S1 without any true match: {(N_TRUE == 0).mean():.1%}")

## Step 7 — XGBoost on the GPU (out-of-fold)

One model per fold, trained on the other folds with early stopping on 10 % of their S1 records, predicting its own fold → honest out-of-fold probabilities for every training pair. Data is streamed to the GPU in batches (`QuantileDMatrix`), so GPU memory stays small. The fold models are averaged to score the test set.

In [ ]:
XGB_PARAMS = {
    "objective": "binary:logistic", "eval_metric": "logloss", "tree_method": "hist", "device": DEVICE,
    "max_depth": 8, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5,
    "reg_lambda": 1.0, "max_bin": 256, "seed": SEED, "verbosity": 0,
    "monotone_constraints": "(" + ",".join("1" if f in MONOTONE else "0" for f in FEATURES) + ")",
}


class Batches(xgb.DataIter):
    def __init__(self, X: np.ndarray, y: np.ndarray, rows: np.ndarray, batch: int = 2_000_000):
        self.X, self.y, self.rows, self.batch, self.k = X, y, rows, batch, 0
        super().__init__()

    def next(self, input_data) -> bool:
        if self.k >= len(self.rows):
            return False
        sl = self.rows[self.k:self.k + self.batch]
        input_data(data=self.X[sl], label=self.y[sl].astype(np.float32))
        self.k += self.batch
        return True

    def reset(self) -> None:
        self.k = 0


class Progress(xgb.callback.TrainingCallback):
    def __init__(self, desc: str, total: int):
        self.pb = tqdm(total=total, desc=desc, unit="round")

    def after_iteration(self, model, epoch, evals_log) -> bool:
        self.pb.update(1)
        loss = evals_log.get("es", {}).get("logloss")
        if loss:
            self.pb.set_postfix(logloss=f"{loss[-1]:.5f}", refresh=False)
        return False

    def after_training(self, model):
        self.pb.close()
        return model


def train_fold(train_rows: np.ndarray, desc: str) -> xgb.Booster:
    rng = np.random.default_rng(SEED)
    groups = np.unique(PI[train_rows])
    held = np.isin(PI[train_rows], rng.choice(groups, max(1, len(groups) // 10), replace=False))
    dtr = xgb.QuantileDMatrix(Batches(X, y, train_rows[~held]), max_bin=XGB_PARAMS["max_bin"])
    des = xgb.QuantileDMatrix(Batches(X, y, train_rows[held]), ref=dtr)
    return xgb.train(XGB_PARAMS, dtr, num_boost_round=3000, evals=[(des, "es")], early_stopping_rounds=100,
                     verbose_eval=False, callbacks=[Progress(desc, 3000)])


def predict(booster: xgb.Booster, X: np.ndarray, rows: np.ndarray | None = None, batch: int = 2_000_000) -> np.ndarray:
    rows = np.arange(len(X)) if rows is None else rows
    out = np.empty(len(rows), dtype=np.float64)
    for s in range(0, len(rows), batch):
        out[s:s + batch] = booster.inplace_predict(X[rows[s:s + batch]], iteration_range=(0, booster.best_iteration + 1))
    return out

In [ ]:
OOF = np.zeros(len(y))
BOOSTERS = []
for k in range(N_FOLDS):
    t = time.time()
    booster = train_fold(np.flatnonzero(PAIR_FOLD != k), f"fold {k + 1}/{N_FOLDS}")
    test_rows = np.flatnonzero(PAIR_FOLD == k)
    OOF[test_rows] = predict(booster, X, test_rows)
    BOOSTERS.append(booster)
    log(f"fold {k + 1}/{N_FOLDS}: {booster.best_iteration + 1} rounds [{time.time() - t:.0f}s]")

gain = pd.Series(BOOSTERS[0].get_score(importance_type="gain")).sort_values(ascending=False)
(gain / gain.sum()).head(15).round(4)

## Step 8 — Decision policy and cross-validated F0.5

The metric is F0.5 averaged **per S1 record** (precision-weighted; a wrong match on a singleton scores 0), so the decision layer matters:

* **Platt calibration** of the out-of-fold probabilities.
* **Exclusivity** — an S2/S3 record belongs to at most one S1 record (S1 is deduplicated): `none`, `hard` (only the best S1 keeps it) or `soft` (ownership as one categorical choice among competing S1 records plus "no owner": qᵢ = oᵢ / (1 + Σ oⱼ), o = p / (1 − p)).
* **Threshold + margin**: predict pairs with q ≥ τ and q ≥ (best q of that S1) − δ; empty prediction when nothing qualifies (singletons).

The policy is chosen on K−1 folds' out-of-fold predictions and scored on the held-out fold (nested), so the reported F0.5 is out-of-sample for model and thresholds.

In [ ]:
TAUS = np.round(np.arange(0.05, 0.96, 0.05), 2)
DELTAS = (0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0)


def calibrate(p: np.ndarray, y_: np.ndarray | None = None, ab: tuple[float, float] | None = None):
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    if ab is None:
        rows = np.random.default_rng(SEED).choice(len(z), min(len(z), 2_000_000), replace=False)
        lr = LogisticRegression(C=1e4, max_iter=1000).fit(z[rows, None], y_[rows])
        ab = (float(lr.coef_[0, 0]), float(lr.intercept_[0]))
    return 1 / (1 + np.exp(-(ab[0] * z + ab[1]))), ab


def exclusivity(q: np.ndarray, pj: np.ndarray, mode: str, n23: int) -> np.ndarray:
    if mode == "none":
        return q
    if mode == "hard":
        order = np.lexsort((-q, pj))
        first = np.r_[True, pj[order][1:] != pj[order][:-1]]
        keep = np.zeros(len(q), dtype=bool)
        keep[order[first]] = True
        return np.where(keep, q, 0.0)
    o = np.clip(q, 0, 1 - 1e-6)
    o = o / (1 - o)
    return o / (1 + np.bincount(pj, weights=o, minlength=n23)[pj])


def select(pi: np.ndarray, q: np.ndarray, tau: float, delta: float, n1: int) -> np.ndarray:
    """Indices of predicted pairs (pairs must be sorted by S1 index)."""
    idx = np.flatnonzero(q >= tau)
    if len(idx) == 0:
        return idx
    g, s = pi[idx], q[idx]
    starts = np.flatnonzero(np.r_[True, g[1:] != g[:-1]])
    gmax = np.maximum.reduceat(s, starts)
    gmax_pair = np.repeat(gmax, np.diff(np.r_[starts, len(s)]))
    return idx[s >= gmax_pair - delta]


def f05(pi: np.ndarray, sel: np.ndarray, label: np.ndarray, n_true: np.ndarray, active: np.ndarray) -> float:
    n = len(n_true)
    k = np.bincount(pi[sel], minlength=n)
    tp = np.bincount(pi[sel][label[sel]], minlength=n)
    f = np.where(tp > 0, 1.25 * tp / np.maximum(k + 0.25 * n_true, 1e-9), 0.0)
    f = np.where(n_true == 0, (k == 0).astype(np.float64), f)
    return float(f[active].mean())


def fit_policy(q_modes: dict, active: np.ndarray) -> tuple[dict, float]:
    best, best_score = None, -1.0
    for mode, q in q_modes.items():
        sub = np.flatnonzero(q >= TAUS.min())  # pairs below the lowest threshold can never be predicted
        pi_s, q_s, y_s = PI[sub], q[sub], y[sub]
        for tau in TAUS:
            for delta in DELTAS:
                sc = f05(pi_s, select(pi_s, q_s, tau, delta, N1), y_s, N_TRUE, active)
                if sc > best_score + 1e-9:
                    best, best_score = {"exclusivity": mode, "tau": float(tau), "delta": float(delta)}, sc
    return best, best_score


def full_report(sel: np.ndarray, active: np.ndarray) -> dict:
    k = np.bincount(PI[sel], minlength=N1)
    tp = np.bincount(PI[sel][y[sel]], minlength=N1)
    single = active & (N_TRUE == 0)
    return {"f05": f05(PI, sel, y, N_TRUE, active),
            "pair_precision": float(tp[active].sum() / max(k[active].sum(), 1)),
            "pair_recall": float(tp[active].sum() / max(N_TRUE[active].sum(), 1)),
            "singleton_accuracy": float((k[single] == 0).mean()) if single.any() else 1.0,
            "false_merge_entities": int((active & (tp < k)).sum())}

In [ ]:
Q_CAL, CALIB = calibrate(OOF, y)
Q_MODES = {m: exclusivity(Q_CAL, PJ, m, N23) for m in ("none", "hard", "soft")}

cv_sel = []
for k in range(N_FOLDS):
    policy_k, _ = fit_policy(Q_MODES, FOLD != k)
    sel_k = select(PI, Q_MODES[policy_k["exclusivity"]], policy_k["tau"], policy_k["delta"], N1)
    cv_sel.append(sel_k[FOLD[PI[sel_k]] == k])
    log(f"fold {k + 1}: policy {policy_k} -> held-out F0.5 {f05(PI, sel_k, y, N_TRUE, FOLD == k):.4f}")
cv_sel = np.concatenate(cv_sel)
ALL = np.ones(N1, dtype=bool)
REPORT = {"cv": full_report(cv_sel, ALL),
          "per_country": {str(c): round(f05(PI, cv_sel, y, N_TRUE, COUNTRY1 == ci), 4) for ci, c in enumerate(COUNTRY_NAMES)},
          "per_fold": [round(f05(PI, cv_sel, y, N_TRUE, FOLD == k), 4) for k in range(N_FOLDS)],
          "candidate_recall": candidate_recall(tr_pairs, tr_gt, N1, N23)}
POLICY, _ = fit_policy(Q_MODES, ALL)
REPORT["final_policy"], REPORT["calibration"] = POLICY, CALIB
log(f"CV F0.5 = {REPORT['cv']['f05']:.4f} | per country {REPORT['per_country']} | final policy {POLICY}")
json.dump(REPORT, open(os.path.join(OUT_DIR, "validation_report.json"), "w"), indent=2)
REPORT

In [ ]:
# free training memory before the test set
del X, y, OOF, Q_CAL, Q_MODES, tr_pairs, tr_enc1, tr_enc23, tr_s1, tr_s23, PI, PJ, PAIR_FOLD
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 9 — Test set

Same pipeline: normalise → embed (training IDF and projection) → retrieve → features → average of the fold models → calibration → exclusivity → threshold / margin.

In [ ]:
te_s1, te_s23, te_is_s3 = load_split("test")
M1, M23 = len(te_s1), len(te_s23)
log(f"test: {M1:,} S1 | {M23:,} S2+S3 | countries {te_s1.country.value_counts().head(6).to_dict()}")
te_n1 = normalize_frame(te_s1, "normalise test S1")
te_n23 = normalize_frame(te_s23, "normalise test S2+S3")
te_s1, te_s23 = te_s1[["entity_id"]], te_s23[["entity_id"]]
te_e1, te_e23 = embed_split(te_n1, te_n23, IDF, "test")
te_pairs = retrieve(te_e1, te_e23, te_is_s3, "test")
del te_e1, te_e23
te_enc1, te_enc23 = encode(te_n1, te_n23)
del te_n1, te_n23
gc.collect()
X_te, names_te = build_features(te_enc1, te_enc23, te_pairs, te_is_s3, "test")
assert names_te == FEATURES
p_te = np.mean([predict(b, X_te) for b in tqdm(BOOSTERS, desc="test: predict", unit="model")], axis=0)
q_te, _ = calibrate(p_te, ab=CALIB)
q_te = exclusivity(q_te, te_pairs["j"], POLICY["exclusivity"], M23)
te_sel = select(te_pairs["i"], q_te, POLICY["tau"], POLICY["delta"], M1)
log(f"test: {len(te_pairs['i']):,} candidate pairs, {len(te_sel):,} predicted matches, "
    f"{len(np.unique(te_pairs['i'][te_sel])):,} of {M1:,} S1 records with >= 1 match")

## Step 10 — Write the submission files

`matching_results.tsv` (the scored file) and `candidate_pairs.tsv` (exactly the pairs the model scored). Every test S1 record gets exactly one row; lists are best-first, de-duplicated, and every match is also a candidate.

In [ ]:
def write_lists(path: str, header: tuple[str, str], s1_ids: pd.Series, c_ids: np.ndarray,
                pi: np.ndarray, pj: np.ndarray, score: np.ndarray) -> None:
    order = np.lexsort((pj, -score, pi))
    lists = pd.Series(c_ids[pj[order]]).groupby(pi[order], sort=True).agg(",".join)
    col = pd.Series("", index=range(len(s1_ids)), dtype=object)
    col[lists.index] = lists.to_numpy()
    pd.DataFrame({header[0]: s1_ids.to_numpy(), header[1]: col.to_numpy()}).to_csv(path, sep="\t", index=False)


c_ids = te_s23.entity_id.to_numpy()
codes = te_pairs["i"] * M23 + te_pairs["j"]
assert len(np.unique(codes)) == len(codes), "duplicate candidate pairs"
match_path = os.path.join(OUT_DIR, "matching_results.tsv")
cand_path = os.path.join(OUT_DIR, "candidate_pairs.tsv")
write_lists(match_path, ("source1_entity_id", "matched_entity_ids"), te_s1.entity_id, c_ids,
            te_pairs["i"][te_sel], te_pairs["j"][te_sel], q_te[te_sel])
write_lists(cand_path, ("source1_entity_id", "candidate_entity_ids"), te_s1.entity_id, c_ids,
            te_pairs["i"], te_pairs["j"], q_te)

check = pd.read_csv(match_path, sep="\t", dtype=str, keep_default_na=False)
assert len(check) == M1 and check.source1_entity_id.tolist() == te_s1.entity_id.tolist()
log(f"wrote {match_path} and {cand_path}  (total runtime {(time.time() - T0) / 60:.1f} min)")
check.head(10)